# Task 01 · AML triage — thin client
All logic lives in `src/aml_triage/`; this notebook only runs the package and shows its outputs.
Reference (original) notebook: `reference/01_aml_graph_triage.ipynb`.

In [1]:
import logging, json
from pathlib import Path
import pandas as pd
from aml_triage.cli import run

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "config.yaml").exists())
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(name)s: %(message)s", force=True)
m = run(ROOT / "config" / "config.yaml")          # same as `aml-train`, about 4 min

2026-11-18 17:10:38,092 aml_triage: 5,077,237 transactions in window
2026-11-18 17:10:52,075 aml_triage: graph: 422,648 nodes, 647,316 edges
2026-11-18 17:12:34,491 aml_triage.features: louvain: 23,638 communities in 78s
2026-11-18 17:12:38,267 aml_triage: 515,070 accounts, 19 profile + 12 structural features
2026-11-18 17:12:45,340 aml_triage: profile only: AUPRC 0.265, p@200 0.975, fit 6.9s
2026-11-18 17:12:53,887 aml_triage: structural only: AUPRC 0.100, p@200 0.410, fit 8.5s
2026-11-18 17:13:03,893 aml_triage: profile + structural: AUPRC 0.292, p@200 0.990, fit 10.0s
2026-11-18 17:14:11,299 aml_triage: forward: AUPRC 0.247, lift 27.5x; new accounts lift 8.6x
2026-11-18 17:14:11,405 aml_triage: done in 217s: AUPRC 0.292, top-200 198/200, sha256 d4d1fbf90a2856e4


In [2]:
pd.DataFrame(m["results"]).T.drop(columns="model").astype(float).round(3)

,AUPRC,AUC,p@100,p@200,p@500,p@1000,p@2000,p@3000,fit_s
profile only,0.265,0.885,1.00,0.975,0.638,0.375,0.235,0.187,6.882
structural only,0.100,0.843,0.48,0.410,0.266,0.192,0.137,0.119,8.546
profile + structural,0.292,0.909,1.00,0.990,0.672,0.401,0.254,0.201,10.005


In [3]:
rows = [m["headline"]] + [m["forward"][k] for k in ("transductive_no_community", "forward", "forward_new_accounts")]
pd.DataFrame(rows).set_index("evaluation").round(3)

,accounts,laundering,base rate,AUPRC,lift,p@200,p@1000
evaluation,,,,,,,
transductive (headline),154521,1759,0.011,0.292,25.681,0.990,0.401
"transductive, no community size",154521,1759,0.011,0.289,25.374,0.985,0.401
forward,372954,3357,0.009,0.247,27.470,1.000,0.619
"forward, new accounts",2766,76,0.027,0.237,8.644,0.170,0.057


In [4]:
queue = pd.read_csv(ROOT / "reports" / "queue_top200.csv", index_col=0)
print(f"top-{len(queue)}: {queue.y.sum()} true laundering · "
      f"{(queue.why == 'no simple motif (model-only)').mean():.0%} with no single-motif explanation")
queue.head(10)

top-200: 198 true laundering · 92% with no single-motif explanation


,y,two_cycle,three_cycle,max_fan_out_day,max_fan_in_day,gather_scatter,in_deg,out_deg,usd_in,usd_out,score,why
from_acct,,,,,,,,,,,,
80515A830,1,0.0,0.0,1.0,1.0,0.0,1.0,1.0,63931.81000,77238.293875,0.999360,no simple motif (model-only)
80BEF6450,1,0.0,0.0,1.0,0.0,0.0,0.0,1.0,20673.73803,23647.418030,0.998756,no simple motif (model-only)
8078D10C0,1,0.0,0.0,1.0,2.0,0.0,2.0,1.0,39247.94000,40782.232500,0.998182,no simple motif (model-only)
80D9A31F0,1,0.0,0.0,1.0,0.0,0.0,0.0,1.0,13997.56195,17857.481950,0.998139,no simple motif (model-only)
80B89ED40,1,0.0,0.0,1.0,0.0,0.0,0.0,1.0,6967.57000,12200.030000,0.998134,no simple motif (model-only)
80D45D750,1,0.0,0.0,1.0,0.0,0.0,0.0,1.0,28716.64000,35727.097100,0.998121,no simple motif (model-only)
80FCC1740,1,0.0,0.0,1.0,2.0,0.0,3.0,1.0,61725.71000,27870.780000,0.998086,no simple motif (model-only)
809B70360,1,0.0,0.0,1.0,0.0,0.0,0.0,1.0,11308.33000,21425.640000,0.997909,no simple motif (model-only)
801C0F2B0,1,0.0,0.0,1.0,1.0,0.0,1.0,4.0,30469.29000,37858.524500,0.997785,no simple motif (model-only)


In [5]:
{k: m[k] for k in ("git_commit", "git_dirty", "holdout", "timings")}

{'git_commit': '30d4c31cb12652249a704c7764a4e952c0898059',
 'git_dirty': False,
 'holdout': {'accounts': 154521,
  'laundering': 1759,
  'sha256': 'd4d1fbf90a2856e456fc806f4132c35d5964d398a3aa934fe7d11d4c2abe9d03'},
 'timings': {'load_s': 3.4610977172851562,
  'graph_s': 13.778010129928589,
  'features_s': 106.19103670120239,
  'train_s': 10.005085945129395,
  'forward_s': 66.9160840511322,
  'total_s': 216.67115139961243}}